In [1]:
print('hello, world')

hello, world


In [1]:
import utils.yolo_fun as yolo_fun
import utils.img_fun as img_fun
import os
import pandas as pd
from tqdm import tqdm  
import rasterio
from rasterio.windows import Window
from rasterio.errors import RasterioIOError
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

#valid_tiles = [13, 14, 15, 22, 23, 24, 35, 44, 45, 51, 52, 53, 54, 61, 62, 63, 70, 71, 72, 73, 81, 82]  

orthomosiac_coords = os.path.join('coords', 'yolo_coords.csv')
coords_dir_sin_normalizar = os.path.join('coords', 'labels_sin_normalizar')
coords_dir_normalized = os.path.join('coords', 'labels_normalized')
os.makedirs(coords_dir_sin_normalizar, exist_ok=True)
os.makedirs(coords_dir_normalized, exist_ok=True)

negatives_dir = os.path.join('negatives')
subrecortes_dir = os.path.join('cut_tiles')

os.makedirs(negatives_dir, exist_ok=True)
os.makedirs(subrecortes_dir, exist_ok=True)
#path_doctorado = 'G:\\.shortcut-targets-by-id\\1pYgV5EIk4-LapLNhlCwpQaDAzuqNffXG\\doctorado_albert\\pinguiton\\ortho_dic_5000'
path_doctorado = 'G:\\.shortcut-targets-by-id\\1pYgV5EIk4-LapLNhlCwpQaDAzuqNffXG\\doctorado_albert\\conteo_pinguinos\\recortes'



In [15]:

# PARTE 0: ITERAMOS SOBRE LAS IMÁGENES PARA RECORTARLAS

contador = 0
for img in os.listdir(path_doctorado):
    
    try:
        # PARTE 1: Cargar la imagen y recortarla en imágenes más pequeñas de aproximadamente 500x500 píxeles
        print(f"\n\nRecortando imagen {img}...")
        print('_________________________________________________________')

        img_name = img.split('.')[0]
        

        tiff_file = os.path.join(path_doctorado, img_name + '.tif')


        # Sacamos un diccionario con toda la información de la imagen
        img_info = img_fun.get_img_info(tiff_file)
        WIDTH = img_info["width"]
        HEIGHT = img_info["height"]
        TOP_LEFT = img_info["top_left"]
        BOTTOM_RIGHT = img_info["bottom_right"]
        min_x, max_y = img_info['top_left']
        max_x, min_y = img_info['bottom_right']

        img_fun.crop_tile_into_subrecortes(
            tiff_file = tiff_file, 
            output_dir = subrecortes_dir, 
            coords_csv = orthomosiac_coords,
            tile_size = 640,
            overlap = 128
        )


        # PARTE 2 (OPCIONAL): EXTRACCIÓN DE METADATOS PARA CADA SUBRECORTE INDIVIUAL
        df_orthomosaic = pd.read_csv(orthomosiac_coords, encoding='ISO-8859-1')
        print("Datos Originales: \n", df_orthomosaic.head())

        # Filtrar los puntos que están dentro del tile
        filtered_data = df_orthomosaic[
            (df_orthomosaic['x_center'] >= min_x) & (df_orthomosaic['x_center'] <= max_x) &
            (df_orthomosaic['y_center'] >= min_y) & (df_orthomosaic['y_center'] <= max_y)
        ]
       # Asegúrate de que el directorio exista
        coords_per_tile = os.path.join('coords', 'coords_per_tile')
        os.makedirs(coords_per_tile, exist_ok=True)

        # Guarda el archivo CSV
        filtered_data.to_csv(os.path.join(coords_per_tile, f'coords_{img_name}.csv'), index=False)
        print(f"Datos filtrados guardados en 'coords/coords_per_tile/coords_{img_name}.csv'.")

     
    except RasterioIOError as e:
        print(f"Error al cargar la imagen con rasterio: {e}")
        continue
    except FileNotFoundError as e:
        print(f"Error: {e}")
        continue

    



Recortando imagen recorte_7.tif...
_________________________________________________________
Metadata:
---------
driver: GTiff
dtype: uint8
nodata: None
width: 10195
height: 11420
count: 4
crs: EPSG:4326
transform: | 0.00, 0.00,-59.21|
| 0.00,-0.00,-62.28|
| 0.00, 0.00, 1.00|
blockxsize: 10195
blockysize: 1
tiled: False
interleave: pixel

Coordenadas de las esquinas de la imagen:
TOP LEFT: (-59.21011298020679, -62.28453303051474)
BOTTOM RIGHT: (-59.20485772277679, -62.287273236674736)
Datos Originales: 
    class   x_center   y_center  width  height
0      0 -59.209791 -62.285558     30      30
1      0 -59.209815 -62.285566     30      30
2      0 -59.209862 -62.285563     30      30
3      0 -59.209878 -62.285563     30      30
4      0 -59.209861 -62.285556     30      30
Datos filtrados guardados en 'coords/coords_per_tile/coords_recorte_7.csv'.


Recortando imagen recorte_8.tif...
_________________________________________________________
Metadata:
---------
driver: GTiff
dtype: 

In [16]:
# PARTE 3: ASIGNACIÓN DE LABELS EN TXT A CADA SUBRECORTE

yolo_fun.generar_txt_yolo(
    subrecorte_dir = subrecortes_dir, 
    csv_file = orthomosiac_coords, 
    coords_dir = coords_dir_sin_normalizar
)

Generando archivos .txt: 100%|██████████| 1401/1401 [00:26<00:00, 53.45it/s]

Archivos .txt generados en coords\labels_sin_normalizar


In [ ]:


# Iteramos para normalizar las coordenadas de cara archivo.txt
for file in os.listdir(coords_dir_sin_normalizar):
    df_sin_normalizar = pd.read_csv(os.path.join(coords_dir_sin_normalizar, file), sep=' ', header=None)
    name_subrecorte = os.path.splitext(file)[0]
    subrecorte_file = os.path.join('cut_tiles', f"{name_subrecorte}.tiff")

    print(f"Normalizando archivo {file}...")
    coords_file = os.path.join(coords_dir_sin_normalizar, file)
    output_file = os.path.join(coords_dir_normalized, file)
    
    yolo_fun.normalize_yolo_coords(
        tiff_file = subrecorte_file,
        coords_sin_normalizar = df_sin_normalizar, 
        output_file = output_file, 
    )


In [18]:


# PARTE 4: CLASIFICAR CONJUNTOS DE TRAIN Y VAL

# Listar todas las imágenes y sus respectivos archivos de coordenadas
images = [img for img in os.listdir(subrecortes_dir) if img.endswith('.tiff')]
annotations = [os.path.join(coords_dir_normalized, f'{os.path.splitext(img)[0]}.txt') for img in images]

# Filtrar solo las imágenes que tienen un archivo de coordenadas no vacío
valid_images = []
valid_annotations = []
for img, txt_path in zip(images, annotations):
    if os.path.exists(txt_path) and os.path.getsize(txt_path) > 0:
        valid_images.append(img)
        valid_annotations.append(txt_path)

# Dividir las imágenes y etiquetas en conjuntos de entrenamiento (80%) y validación (20%)
train_images, val_images, train_annotations, val_annotations = train_test_split(
    valid_images, valid_annotations, test_size=0.2, random_state=42
)

# Crear directorios de salida si no existen
os.makedirs('./datasets/penguin_dataset/images/train', exist_ok=True)
os.makedirs('./datasets/penguin_dataset/images/val', exist_ok=True)
os.makedirs('./datasets/penguin_dataset/labels/train', exist_ok=True)
os.makedirs('./datasets/penguin_dataset/labels/val', exist_ok=True)

# Copiar archivos al conjunto de entrenamiento
for img, txt in zip(train_images, train_annotations):
    shutil.copy(os.path.join(subrecortes_dir, img), './datasets/penguin_dataset/images/train')
    shutil.copy(txt, './datasets/penguin_dataset/labels/train')

# Copiar archivos al conjunto de validación
for img, txt in zip(val_images, val_annotations):
    shutil.copy(os.path.join(subrecortes_dir, img), './datasets/penguin_dataset/images/val')
    shutil.copy(txt, './datasets/penguin_dataset/labels/val')

print(f"Se procesaron {len(train_images)} imágenes para entrenamiento y {len(val_images)} imágenes para validación.")

Se procesaron 1120 imágenes para entrenamiento y 281 imágenes para validación.


### Obtención de negativos

Aquí obtendremos imágenes negativas

In [2]:
img = os.path.join(path_doctorado, 'recorte_72.tif')
print(f"\n\nRecortando imagen {img}...")
print('_________________________________________________________')

try:
    # PARTE 1: Cargar la imagen y recortarla en imágenes más pequeñas de aproximadamente 500x500 píxeles
    img_name = img.split('.')[0]
    tiff_file = img

    # Sacamos un diccionario con toda la información de la imagen
    img_info = img_fun.get_img_info(tiff_file)
    WIDTH = img_info["width"]
    HEIGHT = img_info["height"]
    TOP_LEFT = img_info["top_left"]
    BOTTOM_RIGHT = img_info["bottom_right"]
    min_x, max_y = img_info['top_left']
    max_x, min_y = img_info['bottom_right']

    img_fun.crop_tile_into_subrecortes(
        tiff_file = tiff_file, 
        output_dir = negatives_dir, 
        coords_csv = orthomosiac_coords,
        tile_size = 640,
        overlap = 0,
        is_negative = True
    )


except RasterioIOError as e:
    print(f"Error al cargar la imagen con rasterio: {e}")
except FileNotFoundError as e:
    print(f"Error: {e}")

    



Recortando imagen G:\.shortcut-targets-by-id\1pYgV5EIk4-LapLNhlCwpQaDAzuqNffXG\doctorado_albert\conteo_pinguinos\recortes\recorte_72.tif...
_________________________________________________________
Metadata:
---------
driver: GTiff
dtype: uint8
nodata: None
width: 10195
height: 11420
count: 4
crs: EPSG:4326
transform: | 0.00, 0.00,-59.24|
| 0.00,-0.00,-62.30|
| 0.00, 0.00, 1.00|
blockxsize: 10195
blockysize: 1
tiled: False
interleave: pixel

Coordenadas de las esquinas de la imagen:
TOP LEFT: (-59.236389267356785, -62.30371447363474)
BOTTOM RIGHT: (-59.231134009926784, -62.30645467979474)


### Clasificación train y test de negativos

In [19]:

txt_negatives_dir = os.path.join("coords", "negatives")
negatives_dir = os.path.join('negatives')


# Listar todas las imágenes y sus respectivos archivos de coordenadas
images = [img for img in os.listdir(negatives_dir) if img.endswith('.tiff')]
annotations = [txt for txt in os.listdir(txt_negatives_dir) if txt.endswith('.txt')]

print(images)
print(annotations)

# Filtrar solo las imágenes que tienen un archivo de coordenadas no vacío
valid_images = []
valid_annotations = []
for img, txt_path in zip(images, annotations):
    valid_images.append(img)
    valid_annotations.append(txt_path)

# Dividir las imágenes y etiquetas en conjuntos de entrenamiento (80%) y validación (20%)
train_images, val_images, train_annotations, val_annotations = train_test_split(
    valid_images, valid_annotations, test_size=0.2, random_state=42
)

# Copiar y renombrar archivos al conjunto de entrenamiento
for i, (img, txt) in enumerate(zip(train_images, train_annotations)):
    # Generar nuevo nombre con prefijo "negative_n"
    new_img_name = f"negative_{i+1}.tiff"
    new_txt_name = f"negative_{i+1}.txt"
    
    # Copiar y renombrar la imagen
    shutil.copy(os.path.join(negatives_dir, img), os.path.join('./datasets/penguin_dataset/images/train', new_img_name))
    
    # Copiar y renombrar el archivo .txt
    shutil.copy(os.path.join(txt_negatives_dir, txt), os.path.join('./datasets/penguin_dataset/labels/train', new_txt_name))

# Copiar y renombrar archivos al conjunto de validación
for i, (img, txt) in enumerate(zip(val_images, val_annotations)):
    # Generar nuevo nombre con prefijo "negative_n"
    new_img_name = f"negative_{i+1+len(train_images)}.tiff"
    new_txt_name = f"negative_{i+1+len(train_images)}.txt"
    
    # Copiar y renombrar la imagen
    shutil.copy(os.path.join(negatives_dir, img), os.path.join('./datasets/penguin_dataset/images/val', new_img_name))
    
    # Copiar y renombrar el archivo .txt
    shutil.copy(txt, os.path.join('./datasets/penguin_dataset/labels/val', new_txt_name))

print(f"Se procesaron {len(train_images)} imágenes para entrenamiento y {len(val_images)} imágenes para validación.")

['recorte_72_-59.23144071695678_-62.30386804035474.tiff', 'recorte_72_-59.23144071695678_-62.304021607074745.tiff', 'recorte_72_-59.23144071695678_-62.30417517379474.tiff', 'recorte_72_-59.23144071695678_-62.30432874051474.tiff', 'recorte_72_-59.23144071695678_-62.30448230723474.tiff', 'recorte_72_-59.23144071695678_-62.304635873954744.tiff', 'recorte_72_-59.23144071695678_-62.30478944067474.tiff', 'recorte_72_-59.23144071695678_-62.30494300739474.tiff', 'recorte_72_-59.23144071695678_-62.30509657411474.tiff', 'recorte_72_-59.23144071695678_-62.305250140834744.tiff', 'recorte_72_-59.23144071695678_-62.30571084099474.tiff', 'recorte_72_-59.23144071695678_-62.30586440771474.tiff', 'recorte_72_-59.23144071695678_-62.306017974434745.tiff', 'recorte_72_-59.23144071695678_-62.30632510787474.tiff', 'recorte_72_-59.23177062031679_-62.30386804035474.tiff', 'recorte_72_-59.23177062031679_-62.304021607074745.tiff', 'recorte_72_-59.23177062031679_-62.30417517379474.tiff', 'recorte_72_-59.231770620

FileNotFoundError: [Errno 2] No such file or directory: 'recorte_72_-59.23506965391678_-62.30478944067474.txtf'